# Edges, tree-sitter module: guava (Java)
Module: `core-module/edges.py`. Every rule must show the rows it fixes.

**Reference:** javac's parser (`java-ref/ListEdges.java`), parse only, not tree-sitter.

| Edge | What javac lists | Match key |
|---|---|---|
| CONTAINS | class / method and its parent | file + child start line + child name |
| INHERITS | `extends` / `implements` | file + class start line + parent simple name |
| IMPORTS_FROM | `import` lines | file + line |
| CALLS | `a.b()` and `new X()` call sites | file + line + called name |

Each edge is matched at most once. **CALLS is matched by name only**: which method is really called needs a full compile.

In [1]:
import pandas as pd, edge_lab as L
pd.set_option('display.max_colwidth', 90); pd.set_option('display.width', 250)
APP = 'guava'
REPO = L.ROOT / 'apps' / APP
DB = L.ROOT / 'results' / APP / 'runs/iter-003_2026-11-17_2247/crg/graph.db'
files = L.java_files(REPO)
R, errors = L.reference(REPO, files)
print('java files:', len(files), '| javac errors:', errors)
pd.Series({e: len(d) for e, d in R.items()}, name='expected (javac)')

java files: 3267 | javac errors: 0


CONTAINS         71080
INHERITS          6231
IMPORTS_FROM     39233
CALLS           301434
Name: expected (javac), dtype: int64

## 1. Module rules

In [2]:
pd.Series(L.E.RULES, name='rule').to_frame()

,rule
1,annotation types (@interface) are classes
2,"anonymous classes (new X() {...}, enum constant bodies) are classes, at the line of th..."
3,annotation members (String value();) are methods
4,"enum constants are calls new Enum(...), at the line of their name"
5,super(...) / this(...) are calls named super / this


## 2. Module with no rules

In [3]:
S0, res0 = L.score_all(R, L.module(REPO, files, rules=frozenset()), L.MOD_KEYS)
S0

,expected (javac),reported,matched,missed,extra (false positive),found %,correct %
CONTAINS,71080.0,66448.0,66448.0,4632.0,0.0,93.5,100.0
INHERITS,6231.0,6231.0,6231.0,0.0,0.0,100.0,100.0
IMPORTS_FROM,39233.0,39233.0,39233.0,0.0,0.0,100.0,100.0
CALLS,301434.0,293707.0,293707.0,7727.0,0.0,97.4,100.0


## 3. Proof per rule
Module run with every rule **except** one, then with all rules. Rows that change = rows that rule fixes.

In [4]:
proof, examples = L.rule_proof(REPO, files, R)
proof

,rule,what,edge,missed -> found,false positives removed
0,1,annotation types (@interface) are classes,CONTAINS,140,0
1,2,"anonymous classes (new X() {...}, enum constant bodies) are classes, at the line of th...",CONTAINS,4444,0
2,3,annotation members (String value();) are methods,CONTAINS,48,0
3,4,"enum constants are calls new Enum(...), at the line of their name",CALLS,6450,0
4,5,super(...) / this(...) are calls named super / this,CALLS,1277,0


In [5]:
examples

,rule,edge,file,line,what,source
0,1,CONTAINS,android/guava-testlib/src/com/google/common/collect/testing/IgnoreJRERequirement.java,30,IgnoreJRERequirement,"@Target({METHOD, CONSTRUCTOR, TYPE, FIELD})"
1,1,CONTAINS,android/guava-testlib/src/com/google/common/collect/testing/features/CollectionFeature...,118,Require,@Retention(RetentionPolicy.RUNTIME)
2,1,CONTAINS,android/guava-testlib/src/com/google/common/collect/testing/features/CollectionSize.java,90,Require,@Retention(RetentionPolicy.RUNTIME)
3,1,CONTAINS,android/guava-testlib/src/com/google/common/collect/testing/features/ListFeature.java,60,Require,@Retention(RetentionPolicy.RUNTIME)
4,1,CONTAINS,android/guava-testlib/src/com/google/common/collect/testing/features/MapFeature.java,87,Require,@Retention(RetentionPolicy.RUNTIME)
5,2,CONTAINS,android/guava-testlib/src/com/google/common/collect/testing/AbstractIteratorTester.java,67,,"new PermittedMetaException(""UnsupportedOperationException or IllegalStateException"") {"
6,2,CONTAINS,android/guava-testlib/src/com/google/common/collect/testing/AbstractIteratorTester.java,75,,"new PermittedMetaException(""UnsupportedOperationException"") {"
7,2,CONTAINS,android/guava-testlib/src/com/google/common/collect/testing/AbstractIteratorTester.java,82,,"new PermittedMetaException(""IllegalStateException"") {"
8,2,CONTAINS,android/guava-testlib/src/com/google/common/collect/testing/AbstractIteratorTester.java,89,,"new PermittedMetaException(""NoSuchElementException"") {"
9,2,CONTAINS,android/guava-testlib/src/com/google/common/collect/testing/AbstractIteratorTester.java,456,,new IteratorOperation() {


## 4. Module with all rules

In [6]:
S1, res1 = L.score_all(R, L.module(REPO, files), L.MOD_KEYS)
pd.concat({'no rules': S0[['found %','missed','extra (false positive)']], 'all rules': S1[['found %','missed','extra (false positive)']]}, axis=1)

no rules                                all rules                              
              found %  missed extra (false positive)   found % missed extra (false positive)
CONTAINS         93.5  4632.0                    0.0     100.0    0.0                    0.0
INHERITS        100.0     0.0                    0.0     100.0    0.0                    0.0
IMPORTS_FROM    100.0     0.0                    0.0     100.0    0.0                    0.0
CALLS            97.4  7727.0                    0.0     100.0    0.0                    0.0

In [7]:
for e in L.EDGES:
    print(e, 'still missed:', int((~res1[e][0].hit).sum()), '| false positives:', int((~res1[e][1].hit).sum()))

CONTAINS still missed: 0 | false positives: 0
INHERITS still missed: 0 | false positives: 0
IMPORTS_FROM still missed: 0 | false positives: 0
CALLS still missed: 0 | false positives: 0
